# FRESEAN Analysis of HEWL (all-atom vs coarse-grained, solution, ~300 K)

Below, we provide an **all-atom and coarse-grained** FRESEAN example using `pyfresean`. The idea is to load a protein trajectory in solution, coarse-grain it, run FRESEAN on the representation (and also AA for comparison), visualize total vibrational densities of states (VDoS), and inspect per-mode spectra contributions for **CG modes 7 and 8**.

We provide some theoretical explanations where relevant, however, one can refer to the original [FRESEAN paper](https://pubs.acs.org/doi/10.1021/acs.jctc.2c01309) for more details.

This notebook needs to be run from the `examples/` directory. It expects HEWL input files under `input_data/HEWL/` (see section 1)

## Setup — paths and analysis parameters

Here, we define some basic parameters for the analysis.
These parameters are shared by the coarse-graining (CG) step and the FRESEAN steps (AA and CG).

- **`START` / `STOP` / `STEP`** — trajectory slice (MDAnalysis-style: `START` inclusive, `STOP` exclusive). Set `STOP = None` to use all frames. The same window is used for CG construction and both analyses.
- **`N_CORR`** — number of frequency bins in the output spectrum (and maximum lag in the correlation FFT).
- **`DT`** — time between saved frames (ps); sets the frequency grid spacing together with `N_CORR`.
- **`SIGMA`** — Gaussian broadening width (cm $^{-1}$) applied in frequency space; used for smoothing the spectrum. Must match between all-atom and CG runs for fair comparison.
- **`N_CONSTRAINTS_AA = 0`** — number of contsriants for a system. Here, we use 0 for the all-atom protein in solution (unlike notebook 01, where $n_{\mathrm{constraints}} = 6$ for a gas-phase isolated molecule).
- **`MODE_NUMBERS = (7, 8)`** — CG modes of interest; commonly used in enhanced sampling protocols.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np

from pyfresean import Align, CoarseGrain, FRESEAN
from pyfresean.postprocess import low_frequency_peaks, mode_spectra, plot_spectra

INPUT_DATA = Path("input_data") / "MD-HEWL-303K"
OUTPUT_DATA = Path("output_data") / "MD-HEWL-303K"
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

# trajectory window (passed to CG construction and both FRESEAN runs)
START = 0        # first frame index (inclusive)
STOP = 5000       # last frame index (exclusive); None = use all frames
STEP = 1         # frame stride between analyzed frames

# FRESEAN spectral parameters (shared by all-atom and CG runs)
N_CORR = 100     # number of frequency bins (= max lag in the correlation FFT)
DT = 0.02        # ps between consecutive trajectory frames
SIGMA = 10.0     # cm^-1 Gaussian broadening applied in frequency space
N_CONSTRAINTS_AA = 0  # solution-phase protein (gas phase would use 6)

# CG modes to inspect (1-based numbering)
MODE_NUMBERS = (7, 8)
MODE_INDICES = tuple(m - 1 for m in MODE_NUMBERS)
MAX_FREQ_PLOT = 200.0  # cm^-1, x-axis limit for VDoS plots

## 1. Loading the all-atom protein trajectory

We read a **protein-only**, PBC-corrected trajectory generated from unbiased NPT MD in water:

- `topol_prot.tpr` — GROMACS topology for the protein selection
- `sample-NPT_prot_pbc.trr` — trajectory after `gmx trjconv -pbc mol`

Velocities in this file are the all-atom $\mathbf{v}_i$ that FRESEAN will mass-weight as $\tilde{\mathbf{v}}_i = \sqrt{m_i}\,\mathbf{v}_i$. These files must be generated with GROMACS ([`run-slurm.sh`](input_data/MD-HEWL-303K/run-slurm.sh) and [`postprocess-prot.sh`](input_data/MD-HEWL-303K/postprocess-prot.sh)) if they are not already present (see [`input_data/README.md`](README.md)).

In [ ]:
# defining the source paths to the all-atom topology and trajectory
topol = INPUT_DATA / "output/topol_prot.tpr"
traj = INPUT_DATA / "output/sample-NPT_prot_pbc.trr"

for path in (topol, traj):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Generate HEWL solution MD first, and then run post-process to generate protein-only files (see markdown above)."
        )

# loading the trajectory and selecting the protein atoms
u_aa = mda.Universe(str(topol), str(traj))
protein = u_aa.select_atoms("protein")

# number of frames to use
n_use = (len(u_aa.trajectory) if STOP is None else STOP) - START
print(f"Protein atoms: {protein.n_atoms}")
print(f"Residues: {protein.n_residues}")
print(f"Trajectory frames: {len(u_aa.trajectory)} (using [{START}:{STOP}:{STEP}], ~{n_use // STEP} frames)")

## 2. Coarse-graining the trajectory

`CoarseGrain.cg_universe` maps each residue to one or two center-of-mass (COM) beads:

- **BACK** — backbone COM
- **SIDE** — sidechain COM (skipped for GLY and terminal caps)

Bead positions and velocities are mass-weighted averages of the constituent atoms. The CG trajectory has fewer degrees of freedom ($3 N_{\mathrm{beads}}$ vs $3 N_{\mathrm{atoms}}$), so high-frequency internal vibrations are not represented explicitly.

`centered_mode="ref"` expresses internal coordinates relative to a **reference** frame (no per-frame tracking). The mapping's `n_constraints` (0 here for solution) is passed through to the CG FRESEAN run.

In [ ]:
# output paths for the coarse-grained topology and trajectory
cg_top = OUTPUT_DATA / "topol-cg.top"
cg_traj = OUTPUT_DATA / "traj-cg.trr"

cg, u_cg = CoarseGrain.cg_universe(
    (topol, traj),
    select="protein",
    start=START,
    stop=STOP,
    step=STEP,
    output_cg_topology=None, # set to None to avoid writing to disk and improve speed
    output_cg_trajectory=None, # set to None to avoid writing to disk and improve speed
    centered_mode="ref",  # reference-frame internal coords (no per-frame tracking)
    in_memory=True,
)

print(f"CG beads: {cg.mapping.n_beads}")
print(f"n_constraints (CG): {cg.mapping.n_constraints}")
print(f"CG frames: {len(u_cg.trajectory)}")

## 3. Run FRESEAN on the coarse-grained trajectory

Before `FRESEAN`, we align each CG frame to the **first-frame** bead positions, using it as a reference. This removes rigid-body motion from the trajectory used in the correlation analysis, as in [Notebook 01](01_AA-alanine-dipeptide-gas-300K.ipynb).

Calling `FRESEAN.run()` follows the same workflow as [Notebook 01](01_AA-alanine-dipeptide-gas-300K.ipynb): collect mass-weighted velocities, build the spectral correlation matrix $\mathbf{C}(\omega)$, and diagonalize at each frequency. We use 0 here for the number of constraints so the VDoS normalization matches the solution-phase CG model.

Results are stored in `analysis_cg.results`: `freqs`, `vdos_total`, `eigenvectors`, `corr_matrix`, etc.

In [ ]:
# aligning CG frames to the first-frame bead positions
u_cg.trajectory[0]
ref_cg = u_cg.atoms.positions.copy()
u_cg.trajectory.add_transformations(
    Align(
        u_cg.atoms,
        reference_positions=ref_cg,
        place_com_in_box=False,
    ),
)

analysis_cg = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
)
analysis_cg.run(start=START, stop=STOP, step=STEP)

freqs_cg = analysis_cg.results.freqs
vdos_cg = analysis_cg.results.vdos_total
eigenvectors_cg = analysis_cg.results.eigenvectors
corr_cg = analysis_cg.results.corr_matrix

## 4. Run FRESEAN on the all-atom trajectory

Here, we run the same FRESEAN workflow on the **full atomic** protein selection, using the same frame window and spectral parameters as section 3.

The all-atom correlation matrix $\mathbf{C}_{\mathrm{AA}}(\omega)$ has dimension $3N_{\mathrm{atoms}} \times 3N_{\mathrm{atoms}}$ at each frequency — much larger than the CG case.

In [ ]:
u_aa.trajectory[0]
ref_aa = protein.positions.copy()
u_aa.trajectory.add_transformations(
    Align(protein, reference_positions=ref_aa, place_com_in_box=False),
)

analysis_aa = FRESEAN(
    u_aa,
    select="protein",
    n_constraints=N_CONSTRAINTS_AA,
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
)
analysis_aa.run(start=START, stop=STOP, step=STEP)

freqs_aa = analysis_aa.results.freqs
vdos_aa = analysis_aa.results.vdos_total

## 5. Plot total VDoS — all-atom vs coarse-grained

The total vibrational density of states for each representation can be compared on the same frequency axis (using `plot_spectra()`):

$$g(\omega) = \sum_k \lambda_k(\omega)$$

for $g_{\mathrm{AA}}$ (black) and $g_{\mathrm{CG}}$ (orange). Vertical lines mark local maxima of $g_{\mathrm{AA}}(\omega)$ below 200 cm $^{-1}$ (using `low_frequency_peaks()`).

One expects qualitative agreement at low frequency — both capture large-scale protein motion — but not a 1:1 amplitude match: CG folds out fast internal vibrations, and the two runs can normalize slightly differently even with matched `N_CORR`, `DT`, and `SIGMA`.

In [ ]:
low_peaks_aa = low_frequency_peaks(freqs_aa, vdos_aa, max_freq=MAX_FREQ_PLOT)
low_peaks_cg = low_frequency_peaks(freqs_cg, vdos_cg, max_freq=MAX_FREQ_PLOT)

print(f"T (all-atom) = {analysis_aa.results.avg_temperature:.1f} K")
print(f"T (CG) = {analysis_cg.results.avg_temperature:.1f} K")
print(f"Low-frequency peaks (cm-1): {freqs_aa[low_peaks_aa]}")

fig, ax = plt.subplots(figsize=(7, 4))
plot_spectra(
    freqs_aa,
    [vdos_aa, vdos_cg],
    ax=ax,
    xlim=(0, MAX_FREQ_PLOT),
    labels=["All-atom VDoS", "CG VDoS"],
    colors=["black", "tab:orange"],
    linestyles=["-", "--"],
    vlines=freqs_aa[low_peaks_aa].tolist(),
    title="Total VDoS: all-atom vs coarse-grained",
)
plt.tight_layout()
plt.show()

## 6. Per-mode VDoS — CG modes 7 and 8

Once $\mathbf{C}(\omega)$ is calculated for the CG run, one can ask how a **single mode** $\mathbf{v}_k$ — taken from the eigenvector matrix at a chosen frequency $\omega_0$ — contributes across the full spectrum **without re-running FRESEAN along projected velocities**:

$$g_k(\omega) = \mathbf{v}_k^{\mathsf T}\,\mathbf{C}_{\mathrm{CG}}(\omega)\,\mathbf{v}_k$$

`mode_spectra(corr_cg, modes)` evaluates this at all frequencies for each mode specified in `modes`.

For enhanced sampling, collective variables often use **modes 7 and 8** at **zero frequency** of the spectrum — as the six lowest modes are mostly global / rigid-body-like translations and rotations. Here we pick those modes at zero frequency and overlay their $g_k(\omega)$ alongside the total $g_{\mathrm{CG}}(\omega)$.

In [ ]:
freq_idx = 0  # zero frequency
cg_modes = eigenvectors_cg[freq_idx, MODE_INDICES]
mode_vdos = mode_spectra(corr_cg, cg_modes)

fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    freqs_cg,
    vdos_cg,
    ax=ax,
    labels="Total CG VDoS",
    colors=["black"],
    linestyles=["-"],
    vlines=freqs_cg[freq_idx],
    vline_colors=["black"],
)
for i, mode_num in enumerate(MODE_NUMBERS):
    plot_spectra(freqs_cg, mode_vdos[i], ax=ax, labels=f"CG mode {mode_num}")
ax.set_xlim(0, MAX_FREQ_PLOT)
ax.set_title(f"CG modes {MODE_NUMBERS} at {freqs_cg[freq_idx]:.1f} cm$^{-1}$")
plt.tight_layout()
plt.show()